In [1]:
import os
import json
import datetime
import warnings
import polars as pl
import pandas as pd
import altair as alt

from src.najdi_rok import najdi_rok
from src.pocet_stran import pocet_stran
from src.bez_bordelu import bez_bordelu
from src.alt_friendly import alt_friendly
from src.hezke_jmeno import hezke_jmeno
from src.kristi_promin import kristi_promin
from src.me_to_neurazi import me_to_neurazi
from src.alt_friendly import alt_friendly

with open(os.path.join('src','kredity.json'), 'r', encoding='utf-8') as kredity:
    kredity = json.loads(kredity.read())
pl.Config(tbl_rows=100)
alt.data_transformers.disable_max_rows()
alt.themes.register('irozhlas', kristi_promin)
alt.themes.enable('irozhlas')
warnings.filterwarnings('ignore')

In [2]:
df = pl.read_parquet(os.path.join("data/cnb_sloupce","100.parquet"))
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","leader.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","008.parquet")), left_on="001", right_on="001", how="left")
df = df.to_pandas()
df = df[df["leader"].str[6].isin(["a", "t"])]
df = df[~df["leader"].str[7].isin(["b", "i", "s", " "])]
df = df[(df["008"].str[15:17] == "xr") & (df["008"].str[35:38] == "cze")]
df = pl.from_pandas(df)
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","022.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","245.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","300.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","655.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","700.parquet")), left_on="001", right_on="001", how="left")
df = df.explode("022_a").filter(pl.col("022_a").is_null())
df = df.with_columns(pl.col('008').map_elements(najdi_rok, return_dtype=int).alias('rok'))
df = df.with_columns(pl.col('300_a').map_elements(pocet_stran, return_dtype=int).alias('stran'))
df = df.with_columns(pl.col('245_a').map_elements(bez_bordelu, return_dtype=str))
df = df.explode('245_p').with_columns(pl.col('245_p').map_elements(bez_bordelu, return_dtype=str))
print(len(df))

716789


In [3]:
aut = pl.read_parquet(os.path.join("data","aut_vyber.parquet"))

In [4]:
df = df.filter(pl.col("rok") >= 1800)

In [5]:
print(len(df))

710142


In [6]:
df = df.filter((~pl.col("245_h").str.contains("grafika")) | pl.col("245_h").is_null()).unique(subset=["008","100_a","245_a","245_p"], keep="first")

In [7]:
print(len(df))

705921


In [8]:
nechcemejetam = [
    "jn20001103401",
    "xx0008006",
    "jn19990008769",
    "jx20060515016",
    "jn19981002230",
    "jn19981001737",
    "jn19990210182",
    "jn19990001842",
    "jn19990002786",
    "jn19990004346",
    "jn20020721077",
    "jn19990210513",
    "jn19990005488",
    "jo20000080627",
    "jn19990000171",
    "jn20001005715",
    "jn19981002409",
    "jn20000810141",
    "jn19981002129",
    "jn20001103529",
    "jn20000810032",
    "jn19990001513",
    "jx20040611003",
    "jn19990005499",
    "jn19981002230",
    "jn19990001907",
    "jo2005267810",
    "jo20241218643",
    "jn20000602144",
    "jn19990005454",
    "jn19981228078",
    "xx0010566",
    "jn20010310318",
    "xx0203193",
    "jn20001227589",
    "jo2003204270",
    "xx0082647",
    "jn19990001239"
]

# df = df.filter(~pl.col("100_7").is_in(nechcemejetam))

In [9]:
df.sample(10)

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64
"""1""","""Luu, Uyen""","""mzk2014837473""","[""aut""]",null,null,null,null,null,"""nkc20142616502""",""" nam a22 a 4500""","""140903s2014 xr a e f 0…",null,null,null,null,null,"""1""","""0""","""Pravá vietnamská kuchyně""","""recepty a příběhy, které na vá…","""Uyen Luu ; fotografie jídla Cl…",null,null,null,null,null,"[""143 s. :""]","[""barev. il. ;""]","[""25 cm""]",null,null,null,"[""7"", ""9""]","[""kuchařské recepty"", ""cookbooks""]","[""fd132687"", null]","[""czenas"", ""eczenas""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2014,143
"""1""","""Dyk, Viktor,""","""jk01030247""","[""aut""]","""1877-1931""",null,null,null,null,"""bk195901421""",""" nam a22 1 4500""","""980316s1959 xr …",null,null,null,null,null,"""1""","""0""","""Milá sedmi loupežníků""","""Buřiči /""","""Viktor Dyk ; k vydání připravi…",null,null,null,null,null,"[""125, [2] s. ;""]",null,"[""12°""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1959,125
"""1""","""Pilzer, Paul Zane,""","""kv2013746292""","[""aut""]","""1954-""",null,null,null,null,"""cpk19980311416""",""" nam a22 a 4500""","""980506s1998 xr e 0…",null,null,null,null,null,"""1""","""0""","""Bůh si přeje ať jste bohatí""",null,"""Paul Zane Pilzer ; [z anglické…",null,null,null,null,null,"[""271 s. ;""]",null,"[""21 cm""]",null,null,null,"[""7"", ""9""]","[""studie"", ""Studies""]","[""fd133597"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1""]","[""Žlábková, Jelena""]","[""trl""]",null,"[""jn20001103035""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1998,271
"""1""","""Jepson, Tim""","""mzk2005300744""","[""aut""]",null,null,null,null,null,"""nkc19971167177""",""" nam a22 a 4500""","""980304s1997 xr a e 0…",null,null,null,null,null,"""1""","""0""","""Řím""",null,"""[Tim Jepson ; český překlad Ir…",null,null,null,null,null,"[""96 s. :""]","[""il. ;""]","[""20 cm +""]","[""1 mapa""]",null,null,"[""7"", ""7""]","[""turistické průvodce"", ""plány měst""]","[""fd133738"", ""fd133027""]","[""czenas"", ""czenas""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1997,96
"""1""","""Šubrt, Filip,""","""jk01131614""","[""aut""]","""1860-1934""",null,null,null,null,"""nkc20091963260""",""" nam a22 a 4500""","""090612s1890 xr g 0…",null,null,null,null,null,"""1""","""0""","""Z pamětí města Deštné a okolí""",null,"""sestavil Filip Šubrt ; doplňky…",null,null,null,null,null,"[""117 s. ;""]",null,"[""17 cm""]",null,null,null,"[""7""]","[""pojednání""]","[""fd133056""]","[""czenas""]",null,null,null,"[""1""]","[""Domečka, Ludvík,""]","[""aut""]","[""1861-1937""]","[""jk01022692""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,nul

In [10]:
df = df.drop_nulls(subset=["700_a","700_4","700_7"]).explode(["700_a","700_4","700_7"])

In [11]:
df.select(pl.col(["700_a","700_4","700_7",'245_a'])).sample(10)

700_a,700_4,700_7,245_a
str,str,str,str
"""Lahoda, Zdeněk,""","""trl""","""jk01071384""","""Vlčí kůže"""
"""Ross, Tony,""","""ill""","""xx0001347""","""Darebák David a tělo"""
"""Znamenaný, Petr,""","""pht""","""xx0124011""","""Plzeň"""
"""Cesnak, Jozef,""","""ill""","""jx20040112005""","""Dějepis pro 6. ročník základní…"
"""Zazula, Roman""","""aut""","""jn20001227545""","""Záchranářské techniky a postup…"
"""Šumberová, Olga,""","""trl""","""jx20040116013""","""Známá tvář"""
"""Klůfová, Petra,""","""trl""","""jo2008416199""","""Nebe mé lásky"""
"""Hauner, Miroslav,""","""aut""","""jz8000333""","""Svařování v otázkách a odpověd…"
"""McRae, Carla""","""ill""","""hka20211107341""","""Mazej ven!"""


## Autorské spolupráce

In [13]:
from itertools import combinations

In [14]:
def find_collaborations(df):
    # Filter for authors only
    authors = df.filter(pl.col('700_4') == 'aut')
    
    # Group by book title to get authors per book
    books_authors = authors.group_by('245_a').agg(pl.col('700_a').alias('authors'))
    
    # Generate author pairs and count collaborations
    collaborations = []
    for book in books_authors.iter_rows():
        if len(book[1]) > 1:  # Only consider books with multiple authors
            for pair in combinations(sorted(book[1]), 2):
                collaborations.append(pair)
    
    # Convert to dataframe and count frequencies
    collab_df = pl.DataFrame({
        'author1': [p[0] for p in collaborations],
        'author2': [p[1] for p in collaborations]
    })
    
    if len(collab_df) == 0:
        return pl.DataFrame({'author1': [], 'author2': [], 'collaboration_count': []})
    
    return (collab_df.group_by(['author1', 'author2'])
            .count()
            .sort('count', descending=True)
            .rename({'count': 'collaboration_count'}))

In [15]:
find_collaborations(df).filter(pl.col("author1") != pl.col("author2"))

author1,author2,collaboration_count
str,str,u32
"""Novotný, Miloš""","""Novák, František""",1165
"""Král, Lukáš""","""Valenta, Tomáš""",1156
"""Krupka, Peter,""","""Nechvátalová, Jana,""",1089
"""Malý, Martin""","""Münch, Otto""",897
"""Münch, Otto""","""Čechová, Jarmila""",826
"""Frydryšková, Yvetta""","""Münch, Otto""",820
"""Krupka, Peter,""","""Staudková, Hana""",792
"""Nechvátalová, Jana,""","""Staudková, Hana""",792
"""Medek, Jaroslav,""","""Petrůj, Svatopluk,""",761


In [16]:
df_autorske = df.filter(pl.col('700_4') == 'aut')

In [17]:
def hezkejmeno(sto):
    if not sto[-1].isalnum():
        sto = sto[:-1]
    if "," in sto:
        sto = sto.split(",")
        sto = sto[1].strip() + " " + sto[0].strip()
    return sto    

In [18]:
df_autorske = df_autorske.with_columns(pl.col("100_a").map_elements(hezkejmeno).alias("jmeno1")).with_columns(pl.col("700_a").map_elements(hezkejmeno).alias("jmeno2"))

In [19]:
df_autorske

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,jmeno1,jmeno2
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str,str
"""1""","""Kopietz, Gerit,""","""jn20001103811""","[""aut""]","""1963-""",null,null,null,null,"""cpk20021184549""",""" nam a22 a 4500""","""021129s2002 xr a c 0…",null,null,null,null,null,"""1""","""0""","""Sídliště v ohrožení""",null,"""Gerit Kopietzová, Jörg Sommer …",null,null,null,null,null,"[""120 s. :""]","[""il. ;""]","[""20 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""publikace pro děti"", ""příběhy"", … ""Children's stories, German""]","[""fd133156"", ""fd133204"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1"", ""1"", ""1""]","""Sommer, Jörg,""","""aut""","[""1963-"", ""1969-"", null]","""jn20001103812""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2002,120,"""Gerit Kopietz""","""Jörg Sommer"""
"""1""","""Lach, Vladimír,""","""jk01071391""","[""aut""]","""1919-""",null,null,null,null,"""ck9202911""",""" nam a22 4500""","""920722s1991 xr a u0…",null,null,null,null,null,"""1""","""0""","""Mikrostruktura stavebních láte…",null,"""Vladimír Lach, Marcela Daňková""",null,null,null,null,null,"[""178 s. :""]","[""obr., tab., grafy, schémata ;""]","[""29 cm""]",null,null,null,"[""7""]","[""učebnice vysokých škol""]","[""fd133772""]","[""czenas""]",null,null,null,"[""1""]","""Daňková, Marcela""","""aut""",null,"""xx0051619""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1991,178,"""Vladimír Lach""","""Marcela Daňková"""
"""1""","""Fischerová-Kvěchová, Marie,""","""jk01031311""","[""ill""]","""1894-1984""",null,null,null,null,"""nkc20182987421""",""" nam a22 i 4500""","""180329s2018 xr a a 0…",null,null,null,null,null,"""1""","""0""","""Malenka""",null,"""obrázky Marie Fischerová-Kvěch…",null,null,null,null,null,"[""50 stran :""]","[""barevné ilustrace ;""]","[""18 x 30 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""česká poezie"", ""obrazové publikace"", … ""children's literature""]","[""fd133958"", ""fd132947"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Malý, Radek,""","""aut""","[""1977-""]","""xx0000314""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2018,50,"""Marie Fischerová-Kvěchová""","""Radek Malý"""
"""1""","""Hrůza, Jiří,""","""jk01043070""","[""aut""]","""1925-2012""",null,null,null,null,"""np9543228""",""" nam a22 4500""","""950615s1995 xr ae e p 0…",null,null,null,null,null,"""1""","""0""","""Vývoj urbanismu I""",null,"""Jiří Hrůza - textová část ; Jo…",null,null,null,null,null,"[""186, 115 s. obr. příl. :""]","[""il., plány ;""]","[""30 cm""]",null,null,null,"[""7""]","[""učebnice vysokých škol""]","[""fd133772""]","[""czenas""]",null,null,null,"[""1""]","""Zajíc, Josef,""","""aut""","[""1943-""]","""ola2003201120""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1995,186,"""Jiří Hrůza""","""Josef Zajíc"""
"""1""","""Sysel, Miroslav""","""xx0013565""","[""aut""]",nul

In [20]:
def kombajmen(jmeno1, jmeno2, radit=True, spojeni = "a"):
    try:
        jmena = [jmeno1.split(" ")[-1], jmeno2.split(" ")[-1]]
        if radit == True:
            jmena.sort()
        return f" {spojeni} ".join(jmena)
    except Exception as e:
        print(e)
        return None

In [21]:
df_autorske = df_autorske.with_columns(pl.struct('jmeno1','jmeno2').map_elements(lambda x: kombajmen(x['jmeno1'], x['jmeno2'])).alias("dvojice"))

In [22]:
df_autorske.group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

dvojice,245_a
str,u32
"""David a Soukup""",126
"""Nedbalová a Nedbalová""",90
"""Engels a Marx""",60
"""Benešová a Kanyzová""",52
"""Benešová a Fukalová""",52
"""Benešová a Příhoda""",52
"""Benešová a Šmíd""",52
"""Benešová a Vondrák""",52
"""Benešová a Formáčková""",52


In [23]:
df_autorske.group_by(['100_a','100_7','700_a']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

100_a,100_7,700_a,245_a
str,str,str,u32
"""David, Petr,""","""xx0006786""","""Soukup, Vladimír,""",126
"""Nedbalová, Marie""","""jo2014816080""","""Nedbalová, Josefína""",87
"""Marx, Karl,""","""jn19990005454""","""Engels, Friedrich,""",59
"""Benešová, Hana,""","""pna2012702164""","""Formáčková, Marie,""",52
"""Benešová, Hana,""","""pna2012702164""","""Fukalová, Lenka""",52
"""Benešová, Hana,""","""pna2012702164""","""Vondrák, Jan,""",52
"""Benešová, Hana,""","""pna2012702164""","""Šmíd, Václav,""",52
"""Benešová, Hana,""","""pna2012702164""","""Kanyzová, Žofie,""",52
"""Benešová, Hana,""","""pna2012702164""","""Příhoda, Pavel,""",52


In [24]:
df.filter(pl.col("700_a") == "Soukup, Vladimír,").filter(pl.col("100_a") == "David, Petr,")

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""cpk20041302392""",""" nam a22 a 4500""","""040521s2004 xr ae e 0…",null,null,null,null,null,"""1""","""0""","""Skvosty Prahy""",null,"""Vladimír Soukup, Petr David, […",null,null,null,null,null,"[""208 s. :""]","[""barev. il., plány ;""]","[""31 cm""]",null,null,null,"[""7"", ""9""]","[""fotografické publikace"", ""photographical works""]","[""fd132276"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1"", ""1""]","""Soukup, Vladimír,""","""aut""","[""1949-"", ""1938-""]","""xx0006793""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2004,208
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""nkc20051627547""",""" cam a22 a 4500""","""050909s2005 xr ab g f 0…",null,null,null,null,null,"""1""","""0""","""Plzeňsko - jih""","""Plzeň /""","""[Petr David, Věra Dobrovolná, …",null,null,null,null,null,"[""191 s. :""]","[""il., mapy ;""]","[""20 cm""]",null,null,null,"[""7"", ""9""]","[""turistické průvodce"", ""tourist guidebooks""]","[""fd133738"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1"", ""1""]","""Soukup, Vladimír,""","""aut""","[""1947-"", ""1949-""]","""xx0006793""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2005,191
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""nkc20223466860""",""" nam a22 i 4500""","""221027s2022 xr ab e f 0…",null,null,null,null,null,"""1""","""0""","""Velký špalíček výletů - 1000 n…",null,"""texty: Petr David, Petr David …",null,null,null,null,null,"[""764 stran :""]","[""barevné ilustrace, mapy ;""]","[""25 cm""]",null,null,null,"[""7"", ""9""]","[""turistické průvodce"", ""tourist guidebooks""]","[""fd133738"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1"", ""1"", ""1""]","""Soukup, Vladimír,""","""aut""","[""1974-"", ""1977-"", ""1949-""]","""xx0006793""",null,null,null,null,"[""ml.,"", null, null]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2022,764
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""np9308201""",""" nam a22 4500""","""940120s1993 xr ab 0…",null,null,null,null,null,"""1""","""0""","""Okolí Prahy""",null,"""[autoři Petr David, Vladimír S…",null,null,null,null,null,"[""127 s. :""]","[""il., mp. ;""]","[""18 cm""]",null,null,null,"[""7""]","[""průvodce""]","[""fd133154""]","[""czenas""]",null,null,null,"[""1"", ""1""]","""Soukup, Vladimír,""","""aut""","[""1949-"", ""1953-""]","""xx0006793""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1993,127
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""cpk20000962539""",""" nam a22 a 4500""","""000502s1999 xr ab e 0…",null,null,null,null,null,"""1""","""0""","""Česká Kanada - Jindřichohradec…",null,"""[Petr David, Vladimír Soukup a…",nul

## Ilustrátorstvo

In [26]:
df_ill = df.filter(pl.col('700_4') == 'ill')

In [27]:
df_ill = df_ill.with_columns(pl.col("100_a").map_elements(hezkejmeno).alias("jmeno1")).with_columns(pl.col("700_a").map_elements(hezkejmeno).alias("jmeno2"))

In [28]:
df_ill = df_ill.with_columns(pl.struct('jmeno1','jmeno2').map_elements(lambda x: kombajmen(x['jmeno1'], x['jmeno2'], radit=False, spojeni="&")).alias("dvojice"))

In [29]:
df_ill.filter(~pl.col("100_7").is_in(nechcemejetam)).group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

dvojice,245_a
str,u32
"""Štíplová & Němeček""",108
"""Nedbalová & Popprová""",87
"""Nedbalová & Poppr""",68
"""Bass & Kratochvíl""",67
"""Švandrlík & Winter-Neprakta""",64
"""Čapek & Čapek""",33
"""Pospíšilová & Trsťan""",31
"""Hašek & Lada""",30
"""Rosecká & Růžička""",28


In [30]:
koliktohobylo = df_ill.unique(subset=['100_7','245_a']).filter(~pl.col("100_7").is_in(nechcemejetam)).filter(pl.col("stran") > 30).group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

In [31]:
dvojice_aut_ill = df_ill.unique(subset=['100_7','245_a']).filter(~pl.col("100_7").is_in(nechcemejetam)).filter(pl.col("stran") > 30).group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True).head(5).select(pl.col("dvojice")).to_series().to_list()
dvojice_aut_ill

['Štíplová & Němeček',
 'Švandrlík & Winter-Neprakta',
 'Pospíšilová & Trsťan',
 'Hašek & Lada',
 'Rosecká & Růžička']

In [32]:
df_ill.filter(~pl.col("100_7").is_in(nechcemejetam)).group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

dvojice,245_a
str,u32
"""Štíplová & Němeček""",108
"""Nedbalová & Popprová""",87
"""Nedbalová & Poppr""",68
"""Bass & Kratochvíl""",67
"""Švandrlík & Winter-Neprakta""",64
"""Čapek & Čapek""",33
"""Pospíšilová & Trsťan""",31
"""Hašek & Lada""",30
"""Rosecká & Růžička""",28


In [33]:
import datetime

In [34]:
df_ill.filter(pl.col("dvojice") == "Pospíšilová & Trsťan")

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,jmeno1,jmeno2,dvojice
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str,str,str
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20183009943""",""" nam a22 i 4500""","""180723s2018 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Kouzelná třída v muzeu""",null,"""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""72 stran :""]","[""barevné ilustrace ;""]","[""23 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2018,72,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20182969572""",""" nam a22 i 4500""","""180111s2018 xr a a 0…",null,null,null,null,null,"""1""","""0""","""Jedeme do školy""","""úkoly pro předškoláky /""","""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""79 stran :""]","[""barevné ilustrace ;""]","[""18 x 25 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""říkadla"", ""publikace pro děti"", … ""children's literature""]","[""fd133992"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2018,79,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20142608897""",""" nam a22 a 4500""","""140714s2014 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Máš nadání na hádání""",null,"""Zuzana Pospíšilová ; ilustrace…",null,null,null,null,null,"[""118 s. :""]","[""barev. il. ;""]","[""23 cm""]",null,null,null,"[""7"", ""9""]","[""publikace pro děti"", ""children's literature""]","[""fd133156"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2014,118,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20112193505""",""" nam a22 a 4500""","""110526s2011 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Letadélko Jurášek""",null,"""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""72 s. :""]","[""barev. il. ;""]","[""23 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,nul

In [35]:
do_grafu = df_ill.filter(
    pl.col("dvojice").is_in(dvojice_aut_ill)
).unique(
    subset=['245_a']
).join(
    koliktohobylo, how="left", on="dvojice"
).with_columns(
    pl.col("245_a_right").map_elements(lambda x: str(x) + "×")
).with_columns(
    pl.concat_str([pl.col('245_a_right'), pl.col('dvojice')], separator=' ').alias('dvojice')
)

In [36]:
dvojice_sort = do_grafu.group_by("dvojice").len().sort(by="len",descending=True).select(pl.col("dvojice")).to_series().to_list()
dvojice_sort

['106× Štíplová & Němeček',
 '62× Švandrlík & Winter-Neprakta',
 '30× Pospíšilová & Trsťan',
 '28× Hašek & Lada',
 '27× Rosecká & Růžička']

In [142]:
df_ill.filter(pl.col("dvojice") == "Štíplová & Němeček")

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,jmeno1,jmeno2,dvojice
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str,str,str
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""bk198003731""",""" nam a22 1 4500""","""960927s1977 xr a a 6 0…",null,null,null,null,null,"""1""","""0""","""Fantóm stadiónu""","""a čtyři další pohádkové příběh…","""Ljuba Štíplová, Jaroslav Němeč…",null,null,null,null,null,"[""34, [2] s. :""]","[""barev. il. ;""]","[""8°""]",null,null,null,null,null,null,null,null,null,null,"[""1""]","""Němeček, Jaroslav,""","""ill""","[""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1977,34,"""Ljuba Štíplová""","""Jaroslav Němeček""","""Štíplová & Němeček"""
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""nkc20081800347""",""" nam a22 a 4500""","""080414s2008 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Muž z budoucnosti""",null,"""napsala Ljuba Štíplová ; nakre…",null,null,null,null,null,"[""90 s. :""]","[""vše barev. il. ;""]","[""31 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""publikace pro děti"", ""komiksy"", … ""comics""]","[""fd133156"", ""fd131978"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Němeček, Jaroslav,""","""ill""","[""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2008,90,"""Ljuba Štíplová""","""Jaroslav Němeček""","""Štíplová & Němeček"""
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""nkc20162783011""",""" nam a22 i 4500""","""160308s2016 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Čtyřlístek a záhadná karta""",null,"""Ljuba Štíplová, Jiří Poborák, …",null,null,null,null,null,"[""64 stran :""]","[""barevné ilustrace ;""]","[""31 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""komiksy"", ""publikace pro děti"", … ""children's literature""]","[""fd131978"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1"", ""1""]","""Němeček, Jaroslav,""","""ill""","[""1942-"", ""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2016,64,"""Ljuba Štíplová""","""Jaroslav Němeček""","""Štíplová & Němeček"""
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""bk198002861""",""" nam a22 1i 4500""","""960720s1979 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Neptun jde ke dnu a čtyři dalš…",null,"""Ljuba Štíplová, Jaroslav Němeč…",null,null,null,null,null,"[""34 stran :""]","[""barevné ilustrace ;""]","[""24 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""komiksy"", ""publikace pro děti"", … ""children's literature""]","[""fd131978"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Němeček, Jaroslav,""","""ill""","[""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,n

In [37]:
df_ill.filter(pl.col("dvojice") == "Pospíšilová & Trsťan")

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,jmeno1,jmeno2,dvojice
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str,str,str
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20183009943""",""" nam a22 i 4500""","""180723s2018 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Kouzelná třída v muzeu""",null,"""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""72 stran :""]","[""barevné ilustrace ;""]","[""23 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2018,72,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20182969572""",""" nam a22 i 4500""","""180111s2018 xr a a 0…",null,null,null,null,null,"""1""","""0""","""Jedeme do školy""","""úkoly pro předškoláky /""","""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""79 stran :""]","[""barevné ilustrace ;""]","[""18 x 25 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""říkadla"", ""publikace pro děti"", … ""children's literature""]","[""fd133992"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2018,79,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20142608897""",""" nam a22 a 4500""","""140714s2014 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Máš nadání na hádání""",null,"""Zuzana Pospíšilová ; ilustrace…",null,null,null,null,null,"[""118 s. :""]","[""barev. il. ;""]","[""23 cm""]",null,null,null,"[""7"", ""9""]","[""publikace pro děti"", ""children's literature""]","[""fd133156"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2014,118,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20112193505""",""" nam a22 a 4500""","""110526s2011 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Letadélko Jurášek""",null,"""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""72 s. :""]","[""barev. il. ;""]","[""23 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,nul

In [38]:
dvojice_sort

['106× Štíplová & Němeček',
 '62× Švandrlík & Winter-Neprakta',
 '30× Pospíšilová & Trsťan',
 '28× Hašek & Lada',
 '27× Rosecká & Růžička']

In [159]:
base_ill = alt.Chart(
    alt_friendly(do_grafu), 
    title=alt.TitleParams(f"{len(dvojice_aut_ill)} nejčastějších dvojic autor/ka-ilustrátor/ka (bez reprintů)")).mark_circle(size=7, filled=True) 

tecky_ill = base_ill.encode(
            x=alt.X("rok:T", title=None, axis=alt.Axis(domainOpacity=0, tickColor='#DCDDD6')), 
            y=alt.Y("dvojice:N", sort=dvojice_sort, title=None, axis=alt.Axis(orient='left', domainOpacity=0, tickColor='white', labelExpr='split(datum.label, "× ")[1]')), 
            yOffset=alt.YOffset("jitter:Q", scale=alt.Scale(range=[3, 15])), 
            color=alt.Color('dvojice:N', scale=alt.Scale(range=['#E09DA3']), 
                            sort=dvojice_sort).legend(None)) \
        .transform_calculate(jitter="sqrt(-2*log(random()))*cos(2*PI*random())")

pocty_ill = base_ill.mark_text().encode(x=alt.X('rok:T', title=None), 
    y=alt.Y('dvojice:N', title=None, sort=dvojice_sort, axis=alt.Axis(orient="right", tickColor='white', labelExpr='split(datum.label, " ")[0]')))

zebricek_ill = alt.layer(tecky_ill, pocty_ill).configure_view(stroke='transparent').properties(
    width=kredity['sirka'] * 1.25, 
    autosize={'type': 'fit', 'contains': 'padding'}
)

zebricek_ill

alt.LayerChart(...)

In [40]:
do_grafu.filter(pl.col('100_a').str.contains('Hašek')).select(pl.col("245_a")).to_series().to_list()

['Veselé povídky',
 'Aféra s křečkem a jiné povídky',
 'Hašek v kostce',
 'Zpověď starého mládence',
 'Oslí historie, aneb, Vojenské články do čítanek',
 'Postrach domu',
 'Humoresky',
 'Osudy dobrého vojáka Švejka za světové války',
 'Dekameron humoru a satiry',
 'Povídky',
 'Potměšilé historie',
 'Osudy dobrého vojáka Švejka',
 'Má drahá přítelkyně Julča',
 'Nešťastný policejní ředitel',
 'Všivá historie a jiné humoresky',
 'Dva tucty povídek',
 'Když kvetou třešně a jiné humoresky',
 'Průvodčí cizinců a jiné satiry z cest i z domova',
 'Malá zoologická zahrada',
 'Švejk před světovou válkou, Velitelem města Bugulmy a další příběhy',
 'Za války i za sovětů v Rusku',
 'Smějeme se s Jaroslavem Haškem',
 'Procházka přes hranice',
 'Ze staré droguerie',
 'Turista Aratáš a jiné humoresky',
 'Když bolševici zrušili Vánoce',
 'Pod věchýtkem humoru',
 'Útrapy vychovatele',
 'Dobrý voják Švejk před válkou a jiné podivné historky',
 'Reelní podnik']

In [161]:
me_to_neurazi(zebricek_ill, soubor="02_psali_ilustrovali", kredity=kredity['default'])

<figure>
    <a href="https://data.irozhlas.cz/knihy-grafy/02_psali_ilustrovali.svg" target="_blank">
    <img src="https://data.irozhlas.cz/knihy-grafy/02_psali_ilustrovali.svg" width="100%" alt="Omlouváme se, ale alternativní text se nepodařilo vygenerovat. Texty v grafu by měly být čitelné ze zdrojového souboru SVG." />
    </a>
    </figure>


In [42]:
df_ill.group_by(['100_a','100_7','700_a']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

100_a,100_7,700_a,245_a
str,str,str,u32
"""Štíplová, Ljuba,""","""jk01131441""","""Němeček, Jaroslav,""",108
"""Nedbalová, Marie""","""jo2014816080""","""Popprová, Andrea""",87
"""Nedbalová, Marie""","""jo2014816080""","""Poppr, Roman""",68
"""Bass, Eduard,""","""jk01011066""","""Kratochvíl, Zdeněk,""",67
"""Švandrlík, Miloslav,""","""jk01131832""","""Winter-Neprakta, Jiří,""",64
"""Wilson, Jacqueline,""","""jn20010310318""","""Sharratt, Nick,""",52
"""Delahaye, Gilbert,""","""xx0203193""","""Marlier, Marcel,""",51
"""Verne, Jules,""","""jn19990008769""","""Benett, Léon,""",50
"""Brezina, Thomas,""","""jn20001227589""","""Fearn, Naomi,""",43


In [43]:
ctyrlistek_leader = " <leader>     cam a22      a 4500"
ctyrlistek_leader = ctyrlistek_leader.split(">")[1]
print(ctyrlistek_leader[6])
print(ctyrlistek_leader[7])

a
m


In [44]:
df.filter(pl.col('700_4') == 'ill').filter(~pl.col("100_7").is_in(nechcemejetam)).group_by(['100_a','100_7','700_a']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True).rename({'100_a':'autor/ka','700_a':'ilustrátor/ka','245_a':'společných knih'}).drop("100_7")

autor/ka,ilustrátor/ka,společných knih
str,str,u32
"""Štíplová, Ljuba,""","""Němeček, Jaroslav,""",108
"""Nedbalová, Marie""","""Popprová, Andrea""",87
"""Nedbalová, Marie""","""Poppr, Roman""",68
"""Bass, Eduard,""","""Kratochvíl, Zdeněk,""",67
"""Švandrlík, Miloslav,""","""Winter-Neprakta, Jiří,""",64
"""Čapek, Karel,""","""Čapek, Josef,""",33
"""Pospíšilová, Zuzana,""","""Trsťan, Drahomír,""",31
"""Hašek, Jaroslav,""","""Lada, Josef,""",29
"""Rosecká, Zdena""","""Růžička, Jiří""",28


In [45]:
df.filter(pl.col('700_4') == 'trl').group_by(['100_a','100_7','700_a']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

100_a,100_7,700_a,245_a
str,str,str,u32
"""Unger, Gert F.,""","""jn20001103529""","""Butala, Tomáš,""",311
"""Kirby, John""","""jx20040611003""","""Pavka, Marek,""",182
"""Courths-Mahler, Hedwig,""","""jn19990001513""","""Schönová, Zuzana""",120
"""Scott, William,""","""ola2003188680""","""Butala, Tomáš,""",101
"""Birkner-Mahler, Frieda,""","""jn20000600873""","""Lacinová, Libuše,""",80
"""Unger, Gert F.,""","""jn20001103529""","""Schönová, Zuzana""",80
"""Unger, Gert F.,""","""jn20001103529""","""Sedláčková, Libuše""",77
"""Pratchett, Terry,""","""jo20000080627""","""Kantůrek, Jan,""",57
"""Brezina, Thomas,""","""jn20001227589""","""Steidlová, Dagmar,""",55


In [46]:
df.filter(pl.col('700_4') == 'ill').group_by('700_a').agg(pl.col("100_a").n_unique()).sort(by="100_a",descending=True)

700_a,100_a
str,u32
"""Born, Adolf,""",149
"""Bouda, Cyril,""",114
"""Burian, Zdeněk,""",112
"""Krejčová, Zdeňka,""",98
"""Svolinský, Karel,""",93
"""Aleš, Mikoláš,""",80
"""Zmatlíková, Helena,""",80
"""Petráček, Jiří,""",78
"""Jiránek, Vladimír,""",75


In [47]:
nejaktivnejsi_ilustratori = df.filter(pl.col("stran") > 30).filter(pl.col('700_4') == 'ill').group_by('700_7').agg(pl.struct(["100_a","245_a"]).n_unique()).sort(by="100_a",descending=True).head(11).select(pl.col("700_7")).to_series().to_list()
nejaktivnejsi_ilustratori = [x for x in nejaktivnejsi_ilustratori if x != None]
nejaktivnejsi_ilustratori

['jk01012660',
 'jk01020396',
 'jk01083186',
 'jk01083128',
 'jk01012795',
 'jk01063265',
 'jk01152754',
 'jk01021645',
 'jk01132366',
 'ola2003162788']

In [48]:
def hezkejmeno(sto):
    if not sto[-1].isalnum():
        sto = sto[:-1]
    if "," in sto:
        sto = sto.split(",")
        sto = sto[1].strip() + " " + sto[0].strip()
    return sto    

In [49]:
do_grafu2 = df.filter(pl.col('700_7').is_in(nejaktivnejsi_ilustratori)).with_columns(pl.col('700_a').map_elements(hezkejmeno)).with_columns(pl.col("rok").map_elements(lambda x: datetime.date(year=int(x), month=1, day=1), return_dtype=pl.Date).cast(pl.Datetime))
nejaktivnejsi_ilustratori2 = do_grafu2.group_by('700_a').agg(pl.struct(["100_a","245_a"]).n_unique()).sort(by="100_a",descending=True).head(10).select(pl.col("700_a")).to_series().to_list()
mrtvi_ilustratori = aut.explode("100_7").filter(pl.col('100_7').is_in(nejaktivnejsi_ilustratori)).explode("046_g").with_columns(pl.col("046_g").map_elements(lambda x: int(x)).alias("umrti")).select(pl.col(["100_7","umrti"])).filter(pl.col('umrti').is_between(1800,2025)).with_columns(pl.col("umrti").map_elements(lambda x: datetime.date(year=int(x), month=1, day=1), return_dtype=pl.Date).cast(pl.Datetime))
do_grafu2 = do_grafu2.join(mrtvi_ilustratori, how='left', left_on='700_7', right_on='100_7').join(
    do_grafu2.group_by('700_7').len(), on='700_7', how='left'
).with_columns(
    pl.col("len").map_elements(lambda x: str(x) + "×")
).with_columns(
    pl.concat_str([pl.col('len'), pl.col('700_a')], separator=' ').alias('700_a')
)
nejaktivnejsi_ilustratori2 = do_grafu2.group_by('700_a').len().sort(by='len',descending=True).select(pl.col('700_a')).to_series().to_list()

In [50]:
mrtvi_ilustratori

100_7,umrti
str,datetime[μs]
"""jk01012660""",2016-01-01 00:00:00
"""jk01012795""",1984-01-01 00:00:00
"""jk01020396""",1981-01-01 00:00:00
"""jk01021645""",1936-01-01 00:00:00
"""jk01132366""",1917-01-01 00:00:00
"""jk01083186""",2011-01-01 00:00:00
"""jk01152754""",2005-01-01 00:00:00


In [51]:
nejaktivnejsi_ilustratori2

['457× Zdeněk Burian',
 '451× Helena Zmatlíková',
 '445× Adolf Born',
 '254× Antonín Šplíchal',
 '252× Cyril Bouda',
 '233× Jiří Winter-Neprakta',
 '204× Věnceslav Černý',
 '198× Zdeňka Krejčová',
 '197× Jaroslav Němeček',
 '169× Karel Ladislav Thuma']

In [52]:
do_grafu2.sample(5)

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,umrti,len
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],datetime[μs],i64,datetime[μs],str
"""1""","""Grey, Zane,""","""jn19990002874""","[""aut""]","""1872-1939""",null,null,null,null,"""bk193601352""",""" nam a22 i 4500""","""990125s1936 xr af g 0…",null,null,null,null,null,"""1""","""0""","""Po neznámé řece džungle""",null,"""Zane Grey ; z angličtiny přelo…",null,null,null,null,null,"[""218 stran, 2 nečíslované složené listy obrazových příloh :""]","[""ilustrace ;""]","[""22 cm""]",null,null,null,"[""7"", ""7""]","[""americké romány"", ""dobrodružné romány""]","[""fd131796"", ""fd132061""]","[""czenas"", ""czenas""]",null,null,null,"[""1"", ""1""]","""457× Zdeněk Burian""","""ill""","[""1875-1962"", ""1905-1981""]","""jk01020396""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1936-01-01 00:00:00,218,1981-01-01 00:00:00,"""457×"""
"""1""","""Kocourek, Vítězslav,""","""jk01060985""","[""aut""]","""1920-1995""",null,null,null,null,"""nkc20142637835""",""" cam a22 a 4500""","""141124s2014 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Za pohádkou kolem světa""","""pohádky národů celého světa /""","""vybral a vypravuje Vítězslav K…",null,null,null,null,null,"[""193 s. :""]","[""barev. il. ;""]","[""25 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""pohádky"", ""publikace pro děti"", … ""children's literature""]","[""fd133054"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""451× Helena Zmatlíková""","""ill""","[""1923-2005""]","""jk01152754""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2014-01-01 00:00:00,193,2005-01-01 00:00:00,"""451×"""
"""1""","""Swift, Jonathan,""","""jn20000605232""","[""aut""]","""1667-1745""",null,null,null,null,"""ck9003004""",""" nam a22 a 4500""","""900411s1990 xr a u0…",null,null,null,null,null,"""1""","""0""","""Gulliverovy cesty""",null,"""Jonathan Swift ; Z angl. přel.…",null,null,null,null,null,"[""336 s. :""]","[""obr., [8] s. barev. obr. ;""]","[""21 cm""]",null,null,null,null,null,null,null,null,null,null,"[""1"", ""1""]","""252× Cyril Bouda""","""ill""","[""1901-1984"", ""1904-1988""]","""jk01012795""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1990-01-01 00:00:00,336,1984-01-01 00:00:00,"""252×"""
"""1""","""Čečetka, František Josef,""","""jk01021086""","[""aut""]","""1871-1942""",null,null,null,null,"""nkc20152705825""",""" cam a22 i 4500""","""150617s2015 xr a g 0…",null,null,null,null,null,"""1""","""0""","""Mistr Jan Hus""",null,"""F.J. Čečetka ; ilustroval: Věn…",null,null,null,null,null,"[""386 stran :""]","[""ilustrace ;""]","[""21 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české romány"", ""biografické romány"", … ""historical novels""]","[""fd133974"", ""fd131905"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1"", ""1""]","""204× Věnceslav Černý""","""ill""","[""1865-1936"", ""1950-""]","""jk01021645""",null,null,null,null,null,null,null,null,null,nu

In [53]:
nejaktivnejsi_ilustratori2

['457× Zdeněk Burian',
 '451× Helena Zmatlíková',
 '445× Adolf Born',
 '254× Antonín Šplíchal',
 '252× Cyril Bouda',
 '233× Jiří Winter-Neprakta',
 '204× Věnceslav Černý',
 '198× Zdeňka Krejčová',
 '197× Jaroslav Němeček',
 '169× Karel Ladislav Thuma']

In [54]:
do_grafu2.sample(3)

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,umrti,len
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],datetime[μs],i64,datetime[μs],str
"""1""","""Trobl, Jaroslav""","""jk01140172""","[""oth""]",null,null,null,null,null,"""bk197601213""",""" nam a22 1 4500""","""970304s1975 xr a 0…",null,null,null,null,null,"""1""","""0""","""Anekdoty od Baltu""",null,"""z litevských originálů vybral …",null,null,null,null,null,"[""92, [2] s. ;""]",null,"[""8°""]",null,null,null,null,null,null,null,null,null,null,"[""1""]","""445× Adolf Born""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1975-01-01 00:00:00,92,2016-01-01 00:00:00,"""445×"""
"""1""","""Jelínek, Jan,""","""jo200120086905""","[""aut""]","""1926-2004""",null,null,null,null,"""ck8402664""",""" nam a22 4500""","""840730s1984 xr |…",null,null,null,null,null,"""1""","""0""","""První Evropan""","""biologický a kult. vývoj člově…","""Jan Jelínek ; fot. M. Tůma, L.…",null,null,null,null,null,"[""56 s. :""]","[""il., mp. ;""]","[""26 cm""]",null,null,null,null,null,null,null,null,null,null,"[""1"", ""1"", ""1""]","""457× Zdeněk Burian""","""ill""","[""1936-"", ""1947-"", ""1905-1981""]","""jk01020396""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1984-01-01 00:00:00,56,1981-01-01 00:00:00,"""457×"""
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""nkc20162783011""",""" nam a22 i 4500""","""160308s2016 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Čtyřlístek a záhadná karta""",null,"""Ljuba Štíplová, Jiří Poborák, …",null,null,null,null,null,"[""64 stran :""]","[""barevné ilustrace ;""]","[""31 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""komiksy"", ""publikace pro děti"", … ""children's literature""]","[""fd131978"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1"", ""1""]","""197× Jaroslav Němeček""","""ill""","[""1942-"", ""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2016-01-01 00:00:00,64,null,"""197×"""


In [163]:
base = alt.Chart(
    do_grafu2.filter(pl.col('700_a').is_in(nejaktivnejsi_ilustratori2)).to_pandas(), title=alt.TitleParams(
        f"{len(nejaktivnejsi_ilustratori2)} nejvydávanějších ilustrátorů a ilustrátorek (bez reprintů)",
    subtitle="Co tečka, to kniha. Černá čárka označuje rok úmrtí."))

kulicky = base.mark_circle(size=7, filled=True).encode(
            x=alt.X("rok:T", title=None, axis=alt.Axis(domainOpacity=0, tickColor='#DCDDD6')), 
            y=alt.Y("700_a:N", sort=nejaktivnejsi_ilustratori2, title=None, axis=alt.Axis(orient='left', domainOpacity=0, tickColor='white', labelExpr='split(datum.label, "× ")[1]' )), #, 
            yOffset=alt.YOffset("jitter:Q", scale=alt.Scale(range=[3, 15])), 
            color=alt.Color('700_a:N', scale=alt.Scale(range=['#E09DA3']), 
                            sort=nejaktivnejsi_ilustratori2).legend(None)) \
        .transform_calculate(jitter="sqrt(-2*log(random()))*cos(2*PI*random())")

kdy_umreli = base.mark_tick(
    color='#292829',  # optional: you can specify color
    thickness=1.5,
    height=9
).encode(
    x=alt.X('umrti:T', title=None),
    y=alt.Y("700_a:N", sort=nejaktivnejsi_ilustratori2, title=None, axis=alt.Axis(orient='left', tickColor='white', labels=False)))

pocty = base.mark_text().encode(x=alt.X('rok:T', title=None), 
    y=alt.Y('700_a:N', title=None, sort=nejaktivnejsi_ilustratori2, axis=alt.Axis(orient="right", tickColor='white', labelExpr='split(datum.label, " ")[0]'))) #

zebricek2 = alt.layer(kulicky, kdy_umreli, pocty).configure_view(stroke='transparent').properties(
    width=kredity['sirka'] * 1.24,
    autosize={'type': 'fit', 'contains': 'padding'}
).resolve_scale(color='independent',x="shared")

zebricek2

alt.LayerChart(...)

In [165]:
me_to_neurazi(zebricek2, kredity=kredity['default'], soubor='02_ilustratorstvo')

<figure>
    <a href="https://data.irozhlas.cz/knihy-grafy/02_ilustratorstvo.svg" target="_blank">
    <img src="https://data.irozhlas.cz/knihy-grafy/02_ilustratorstvo.svg" width="100%" alt="Omlouváme se, ale alternativní text se nepodařilo vygenerovat. Texty v grafu by měly být čitelné ze zdrojového souboru SVG." />
    </a>
    </figure>


In [57]:
df.filter(pl.col('700_a') == 'Born, Adolf,')

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64
"""1""","""Steklač, Vojtěch,""","""jk01121138""","[""aut""]","""1945-2021""",null,null,null,null,"""cpk20011021896""",""" nam a22 aa4500""","""010905m20002001xr a c 0…",null,null,null,null,null,"""1""","""0""","""Boříkovy lapálie""",null,"""Vojtěch Steklač ; ilustroval A…",null,null,null,null,null,"[""2 sv. :""]","[""barev. il. ;""]","[""21 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""publikace pro děti"", ""příběhy"", … ""Children's stories, Czech""]","[""fd133156"", ""fd133204"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Born, Adolf,""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2000,2
"""1""","""Drijverová, Martina,""","""jk01023067""","[""aut""]","""1951-2022""",null,null,null,null,"""cpk20021191046""",""" cam a22 a 4500""","""021127s2002 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Sísa Kyselá a ušmudlaný rytíř""",null,"""Martina Drijverová ; [ilustrov…",null,null,null,null,null,"[""69 s. :""]","[""barev. il. ;""]","[""20 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""příběhy"", ""publikace pro děti"", … ""Children's stories, Czech""]","[""fd133204"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Born, Adolf,""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2002,69
"""1""","""Shakespeare, William,""","""jn19981002129""","[""aut""]","""1564-1616""",null,null,null,null,"""np9430184""",""" nam a22 4500""","""941024s1993 xr a u0…",null,null,null,null,null,"""1""","""0""","""Sen noci svatojánské""",null,"""William Shakespeare ; Přel. [z…",null,null,null,null,null,"[""104 s. :""]","[""obr. ;""]","[""31 cm""]",null,null,null,"[""7"", ""7""]","[""divadelní hry"", ""dramata""]","[""fd132028"", ""fd132072""]","[""czenas"", ""czenas""]",null,null,null,"[""1"", ""1"", … ""1""]","""Born, Adolf,""","""ill""","[null, ""1930-2016"", … null]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1993,104
"""1""","""Macourek, Miloš,""","""jk01072813""","[""aut""]","""1926-2002""",null,null,null,null,"""nkc20152737529""",""" nam a22 i 4500""","""151014s2015 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Mach a Šebestová na cestách""",null,"""Miloš Macourek, Adolf Born""",null,null,null,null,null,"[""108 stran :""]","[""barevné ilustrace ;""]","[""25 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Born, Adolf,""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2015,108
"""1""","""Žáček, Jiří,""","""jk01152946""","[""aut""]","""1945-""",null,null,null,null,"""cp

In [58]:
df.filter(pl.col('700_4') == 'ill').group_by('700_a').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

700_a,245_a
str,u32
"""Born, Adolf,""",281
"""Burian, Zdeněk,""",230
"""Němeček, Jaroslav,""",179
"""Bouda, Cyril,""",166
"""Zmatlíková, Helena,""",164
"""Winter-Neprakta, Jiří,""",160
"""Krejčová, Zdeňka,""",159
"""Šplíchal, Antonín,""",143
"""Thuma, Karel Ladislav,""",134


In [59]:
df_preklady = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","041.parquet")), left_on="001", right_on="001", how="left")

In [60]:
df_preklady.filter(pl.col('700_4') == 'trl').group_by('700_a').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

700_a,245_a
str,u32
"""Butala, Tomáš,""",502
"""Pavka, Marek,""",429
"""Vyskočil, Josef""",344
"""Sladká, Stanislava""",246
"""Schönová, Zuzana""",234
"""Volhejnová, Veronika,""",230
"""Podaný, Richard,""",224
"""Kuťák, Jaroslav,""",214
"""Chromiaková, Eva""",196


In [61]:
df_preklady.filter(pl.col('700_a') == "Butala, Tomáš,").group_by("100_a").len()

100_a,len
str,u32
"""Glaser, Frank""",9
"""Roberts, Dan,""",1
"""Silva, Tony,""",1
"""Robertson, Frank C.""",2
"""Aurel, Catherine,""",1
"""Unger, Gert F.,""",391
"""Morgan, Matt""",4
"""Hayes, Rex,""",13
"""Arnaldur Indriðason,""",5


In [62]:
df_preklady.filter(pl.col('700_a') == "Butala, Tomáš,").group_by("041_h").len()

041_h,len
list[str],u32
"[""fre""]",1
"[""ice""]",5
"[""ger""]",633
null,2
"[""eng""]",3


In [63]:
df_preklady.filter(pl.col('700_4') == 'trl').group_by('700_a').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

700_a,245_a
str,u32
"""Butala, Tomáš,""",502
"""Pavka, Marek,""",429
"""Vyskočil, Josef""",344
"""Sladká, Stanislava""",246
"""Schönová, Zuzana""",234
"""Volhejnová, Veronika,""",230
"""Podaný, Richard,""",224
"""Kuťák, Jaroslav,""",214
"""Chromiaková, Eva""",196


In [64]:
df_preklady.filter(pl.col('700_4') == 'trl').group_by('700_a').agg(pl.col("stran").sum()).sort(by="stran",descending=True)

700_a,stran
str,i64
"""Volhejnová, Veronika,""",104086
"""Pacnerová, Jana,""",90284
"""Kantůrek, Jan,""",80551
"""Podaný, Richard,""",76323
"""Dušek, Zdík,""",68870
"""Jašová, Jana,""",66951
"""Klůfová, Petra,""",66769
"""Chodilová, Dana,""",56060
"""Medek, Pavel,""",54195


In [65]:
df_preklady.filter(pl.col('700_a') == "Volhejnová, Veronika,").group_by("100_a").len().sort(by="len",descending=True)

100_a,len
str,u32
"""Kinney, Jeff,""",52
"""Christie, Agatha,""",29
"""Herbert, Frank,""",24
"""Walliams, David,""",21
"""Lewis, C. S.""",21
"""Green, John,""",17
"""Colfer, Chris,""",16
"""Le Carré, John,""",8
"""Morse, Brian,""",7


In [66]:
df_preklady.explode("041_h").filter(pl.col('700_4') == 'trl').group_by('700_a').agg(pl.col("041_h").n_unique()).sort(by="041_h",descending=True)

700_a,041_h
str,u32
"""Babler, Otto František,""",19
"""Vetti, O. S.,""",16
"""Hiršal, Josef,""",16
"""Bednář, Kamil,""",12
"""Vrchlický, Jaroslav,""",12
"""Fischer, Otokar,""",11
"""Vladislav, Jan,""",11
"""Sýs, Karel,""",11
"""Žáček, Jiří,""",11


In [67]:
df_preklady.filter(pl.col('700_a') == "Babler, Otto František,").group_by('041_h').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

041_h,245_a
list[str],u32
"[""eng""]",10
"[""ger""]",9
"[""fre""]",8
"[""scr""]",5
null,5
"[""hrv""]",3
"[""ita""]",3
"[""srp""]",2
"[""bul""]",2


In [68]:
df_preklady.filter(pl.col('700_a') == "Hiršal, Josef,").group_by('041_h').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

041_h,245_a
list[str],u32
"[""ger""]",32
null,12
"[""por""]",4
"[""spa""]",3
"[""fre""]",2
"[""mul""]",2
"[""swe""]",2
"[""scr""]",2
"[""rum""]",2
